In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)
emb = embedding_factory(
    "openai",
    model="text-embedding-3-small",
    client=client
)

### 공통 구조
1. 서로 다른 프롬프트 2개로 LLM 채점자가 2번 채점
2. [0, 1]로 정규화 후 평균(하나만 유효하면 그 값 그대로 사용)
3. 추론 과정 없이 점수만 내기 때문에 **토큰이 적고**, **작은 모델**에서도 잘 돌아감.
4. 단, 설명력이 낮다.

#### Answer Accuracy

**목적:** 응답이 **정답(reference)과 얼마나 일치하는지** 평가
**핵심 질문:** "응답이 정답과 정확하게 일치하는가?"

- LLM × 2
- Reference 필수
- 점수: **0 / 2 / 4 → 0~1로 정규화**
- 두 개의 평가 템플릿을 사용하여 서로 다른 관점에서 평가

**평가 과정**

1. 첫 번째 LLM이 `response`를 `reference`와 비교하여 평가
2. 두 번째 LLM은 **response와 reference의 역할을 서로 바꿔서** 다시 평가
3. 각각 **0 / 2 / 4점**으로 평가
4. 두 점수를 정규화한 뒤 평균

| 점수 | 의미 |
|---:|---|
| 0 | 부정확하거나 다른 질문에 답함 |
| 2 | 부분적으로 일치 |
| 4 | 정확하게 일치 |

**예시**

질문: **"프랑스의 수도는 어디인가?"**

Reference:

> "프랑스의 수도는 파리이다."

Response:

> "프랑스의 수도는 파리이며, 유럽에서 가장 유명한 도시 중 하나이다."

첫 번째 평가:

- Response → Reference: **4점**
- 정답을 정확하게 포함하고 있음

두 번째 평가:

- Reference → Response: **4점**
- Reference의 핵심 내용이 Response에 포함되어 있음

정규화:

$$
\frac{4}{4} = 1.0
$$

$$
\text{Answer Accuracy}
=
\frac{1.0 + 1.0}{2}
=
\mathbf{1.0}
$$

> 두 평가 결과가 모두 **4점**이면 최종 점수는 **1.0**.

In [ ]:
# 코드
from ragas.metrics.collections import AnswerAccuracy

await AnswerAccuracy(llm=llm).ascore(
    user_input="When was Einstein born?",
    response="Albert Einstein was born in 1879.",
    reference="Albert Einstein was born in 1879."
)

#### Context Relevance

**목적:** 검색된 컨텍스트가 **질문과 얼마나 관련 있는지** 평가
**핵심 질문:** "검색한 내용이 질문에 답하는 데 도움이 되는가?"

- LLM 사용
- 점수: **0 / 1 / 2**
- 2 = 완전 관련
- 1 = 부분 관련
- 0 = 무관
- 각 점수를 2로 나눈 뒤 평균하여 **0~1**로 정규화

**평가 과정**

1. 각 retrieved context가 질문과 관련 있는지 평가
2. 관련성에 따라 0 / 1 / 2점 부여
3. 각 점수를 2로 나눔
4. 전체 평균 계산

$$
\text{Context Relevance}
=
\frac{1}{N}
\sum_{i=1}^{N}
\frac{s_i}{2}
$$

- $N$ = 검색된 컨텍스트 개수
- $s_i \in \{0,1,2\}$ = $i$번째 컨텍스트의 관련성 점수

**예시**

질문: **"프랑스의 수도는 어디인가?"**

검색된 컨텍스트:

1. "프랑스의 수도는 파리이며, 에펠탑이 있다." → **2점**
2. "파리는 프랑스에서 가장 큰 도시 중 하나이다." → **1점**
3. "베를린은 독일의 수도이다." → **0점**

각 점수를 0~1로 변환:

- 2점 → $2/2 = 1.0$
- 1점 → $1/2 = 0.5$
- 0점 → $0/2 = 0.0$

$$
\text{Context Relevance}
=
\frac{1.0 + 0.5 + 0.0}{3}
=
\mathbf{0.5}
$$

> **원 RAGAS 논문:** 문장 단위로 관련성을 판단하여
> `관련 문장 수 / 전체 문장 수`로 정의
>
> **현재 구현:** 전체 컨텍스트를 **0~2점으로 이산 판정**
>
> → 문장 경계 오류에 덜 민감하고 계산량이 적기 때문에 의도적으로 변경

In [ ]:
# 코드
from ragas.metrics.collections import ContextRelevance

await ContextRelevance(llm=llm).ascore(
    user_input="",
    retrieved_contexts=[""]
)

#### Response Groundedness

**목적:** 응답이 검색된 컨텍스트에 **얼마나 근거하고 있는지** 평가
**핵심 질문:** "응답의 내용이 검색된 컨텍스트에 근거하고 있는가?"

- LLM 사용
- 입력: `response`, `retrieved_contexts`
- `user_input`은 사용하지 않음
- 점수: **0 / 1 / 2 → 0~1로 정규화**
- **Faithfulness의 경량 버전**으로 볼 수 있음

| 점수 | 의미 |
|---:|---|
| 0 | 컨텍스트에 근거하지 않음 |
| 1 | 일부만 근거함 |
| 2 | 대부분 또는 완전히 근거함 |

**평가 과정**

1. `response`와 `retrieved_contexts`를 LLM에 전달
2. 응답이 컨텍스트에 얼마나 근거하는지 **0 / 1 / 2점**으로 평가
3. 점수를 2로 나누어 **0~1**로 정규화

$$
\text{Response Groundedness}
=
\frac{\text{평가 점수}}{2}
$$

**예시**

컨텍스트:

> "파리는 프랑스의 수도이며, 인구는 약 210만 명이다."

응답:

> "프랑스의 수도는 파리이며, 인구는 약 210만 명이다."

→ 컨텍스트의 내용에 완전히 근거함 → **2점**

$$
\text{Response Groundedness}
=
\frac{2}{2}
=
\mathbf{1.0}
$$

반대로 응답이:

> "프랑스의 수도는 파리이며, **에펠탑은 1889년에 건설되었다.**"

라면 에펠탑 건설 연도는 컨텍스트에 없으므로 일부만 근거함 → **1점**

$$
\text{Response Groundedness}
=
\frac{1}{2}
=
\mathbf{0.5}
$$

> **Faithfulness와 차이**
>
> - **Faithfulness** → 응답을 **claim 단위로 분해**하고 각각 컨텍스트에 의해 뒷받침되는지 평가
> - **Response Groundedness** → 응답 전체가 컨텍스트에 **얼마나 근거하는지** 간단하게 평가
>
> 따라서 Response Groundedness는 **더 단순하고 가벼운 평가 방식**이라고 볼 수 있음.

In [ ]:
# 코드
from ragas.metrics.collections import ResponseGroundedness

await ResponseGroundedness(llm=llm).ascore(
    response="",
    retrieved_contexts=[""],
)